In [3]:
%pip install langchain-mcp-adapters

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.2/85.2 kB 1.9 MB/s eta 0:00:00 MB/s eta 0:00:01
  Using cached python_multipart-0.0.20-py3-none-any.whl.metadata (1.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.1/173.1 kB 4.4 MB/s eta 0:00:005.0 MB/s eta 0:00:01
Using cached python_multipart-0.0.20-py3-none-any.whl (24 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 kB 4.6 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
from dotenv import load_dotenv 

load_dotenv()

True

In [6]:
import os 
from langchain_mcp_adapters.client import MultiServerMCPClient

github_pat = os.getenv("GITHUB_PAT")

mcp_client = MultiServerMCPClient({
    "github": {
        "url": "https://api.githubcopilot.com/mcp/",
        "headers": {
            "Authorization": f"Bearer {github_pat}"
        },
        "transport": "streamable_http"
    }
})

In [7]:
tool_list = await mcp_client.get_tools()

In [8]:
tool_list

[StructuredTool(name='add_comment_to_pending_review', description="Add review comment to the requester's latest pending pull request review. A pending review needs to already exist to call this (check with the user if not sure).", args_schema={'properties': {'body': {'description': 'The text of the review comment', 'type': 'string'}, 'line': {'description': 'The line of the blob in the pull request diff that the comment applies to. For multi-line comments, the last line of the range', 'type': 'number'}, 'owner': {'description': 'Repository owner', 'type': 'string'}, 'path': {'description': 'The relative path to the file that necessitates a comment', 'type': 'string'}, 'pullNumber': {'description': 'Pull request number', 'type': 'number'}, 'repo': {'description': 'Repository name', 'type': 'string'}, 'side': {'description': 'The side of the diff to comment on. LEFT indicates the previous state, RIGHT indicates the new state', 'enum': ['LEFT', 'RIGHT'], 'type': 'string'}, 'startLine': {'

In [11]:
from langchain.chat_models.base import init_chat_model

llm = init_chat_model(model="gpt-4o-mini", model_provider="openai")

In [12]:
from langchain.agents import create_agent

agent = create_agent(
        model=llm,
        tools=tool_list,
        system_prompt="Use the tools provided to you to answer the user's question"
)

In [13]:
async def process_stream(stream_generator):
    results = []
    try:
        async for chunk in stream_generator:

            key = list(chunk.keys())[0]
            
            if key == 'agent':
                # Agent 메시지의 내용을 가져옴. 메세지가 비어있는 경우 어떤 도구를 어떻게 호출할지 정보를 가져옴
                content = chunk['agent']['messages'][0].content if chunk['agent']['messages'][0].content != '' else chunk['agent']['messages'][0].additional_kwargs
                print(f"'agent': '{content}'")
            
            elif key == 'tools':
                # 도구 메시지의 내용을 가져옴
                for tool_msg in chunk['tools']['messages']:
                    print(f"'tools': '{tool_msg.content}'")
            
            results.append(chunk)
        return results
    except Exception as e:
        print(f"Error processing stream: {e}")
        return results

In [ ]:
from langchain_core.messages import HumanMessage

query = """깃헙의 Pull Request를 확인하고 코드 리뷰를 작성해주세요.

PR URL: https://github.com/ReTrip-Dev/ReTrip-api/pull/42

"""

stream_generator = agent.astream({'message': [HumanMessage(content=query)]})

all_chunks = await process_stream(stream_generator)

if all_chunks:
    final_result = all_chunks[-1]
    print(final_result)